# 05 Chunk Cleaning + Embeddings + FAISS Index

**Input:** `data/processed/chunks/all_chunks.json` (2,240 chunks)
**Output:**
- `data/processed/chunks/clean_chunks.json` - filtered, clean chunks
- `data/vectorstore/henselmans.index` - FAISS index
- `data/vectorstore/henselmans_metadata.json` - chunk metadata aligned with index

**Cleaning fixes (from quality analysis):**
1. Filter TOC/index chunks (lines with 5+ dots in a row)
2. Filter cover page chunks (< 150 chars after stripping page markers)
3. Hard trim chunks > 2200 chars at sentence boundary

**Embedding:**
- Model: `BAAI/bge-m3` (local, free, 1024 dims, 100+ languages)
- Batch size: 16 (safe for CPU)
- FAISS index type: `IndexFlatIP` (inner product = cosine on normalized vectors)

In [6]:
import json, re
import os
from dotenv import load_dotenv
from pathlib import Path
from datetime import datetime
import statistics
import numpy as np
import warnings; warnings.filterwarnings('ignore')

load_dotenv()

from tqdm import tqdm
import faiss
from FlagEmbedding import BGEM3FlagModel

NOTEBOOK_DIR = Path().resolve()
BACKEND_DIR = NOTEBOOK_DIR.parent
PROCESSED_DIR = BACKEND_DIR / 'data' / 'processed'
CHUNKS_DIR = PROCESSED_DIR / 'chunks'
VS_DIR = BACKEND_DIR / 'data' / 'vectorstore'
VS_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_MODEL = os.getenv('EMBEDDING_MODEL')

CHUNKS_PATH = CHUNKS_DIR / 'all_chunks.json'

# Embedding config
BATCH_SIZE = 16
MAX_LENGTH = 512   # BGE-M3 token limit per chunk
EMBED_DIM = 1024  # BGE-M3 output dimension

print('Setup complete')
print(f'Vectorstore dir : {VS_DIR}')

Setup complete
Vectorstore dir : D:\Cybernetic Gym Assistant\backend\data\vectorstore


## 1. Load Chunks

In [2]:
data = json.loads(CHUNKS_PATH.read_text(encoding='utf-8'))
chunks = data['chunks']
print(f'Loaded {len(chunks):,} chunks')

Loaded 2,240 chunks


## 2. Clean Chunks

In [3]:
def is_toc_chunk(text: str) -> bool:
    """Detect table of contents / index pages - multiple lines with 5+ consecutive dots."""
    dot_lines = sum(1 for line in text.split('\n') if '.....' in line)
    return dot_lines >= 4

def is_cover_page_chunk(text: str) -> bool:
    """Detect cover/title page chunks - very short after removing page markers."""
    cleaned = re.sub(r'--- Page \d+ ---', '', text).strip()
    return len(cleaned) < 150

def hard_trim(text: str, max_chars: int = 2200) -> str:
    """Trim chunk to max_chars at the nearest sentence boundary."""
    if len(text) <= max_chars:
        return text
    # Find last sentence end before max_chars
    cutoff = text[:max_chars]
    for sep in ['. ', '? ', '! ', '\n']:
        idx = cutoff.rfind(sep)
        if idx > max_chars * 0.7:   # at least 70% of target
            return cutoff[:idx + 1].strip()
    return cutoff.strip()  # hard cut if no sentence boundary found

# Apply filters
before = len(chunks)

toc_removed = 0
cover_removed = 0
trimmed = 0
clean_chunks = []

for c in chunks:
    text = c['text']

    if is_toc_chunk(text):
        toc_removed += 1
        continue

    if is_cover_page_chunk(text):
        cover_removed += 1
        continue

    if len(text) > 2200:
        c['text'] = hard_trim(text)
        trimmed += 1

    clean_chunks.append(c)

after = len(clean_chunks)

print(f'Cleaning results:')
print(f'  Before: {before:,}')
print(f'  TOC removed: {toc_removed}')
print(f'  Cover removed: {cover_removed}')
print(f'  Trimmed (>2200): {trimmed}')
print(f'  After: {after:,}')
print(f'  Removed total: {before - after}  ({(before-after)/before*100:.1f}%)')

# Re-check lengths
lengths = [len(c['text']) for c in clean_chunks]
print(f'\nClean chunk lengths:')
print(f'  Min: {min(lengths)}')
print(f'  Max: {max(lengths)}')
print(f'  Mean: {statistics.mean(lengths):.0f}')
print(f'  Median: {statistics.median(lengths):.0f}')
print(f'  > 2200: {sum(1 for l in lengths if l > 2200)}')
print(f'  < 150: {sum(1 for l in lengths if l < 150)}')

Cleaning results:
  Before: 2,240
  TOC removed: 106
  Cover removed: 36
  Trimmed (>2200): 881
  After: 2,098
  Removed total: 142  (6.3%)

Clean chunk lengths:
  Min: 212
  Max: 2200
  Mean: 1910
  Median: 2026
  > 2200: 0
  < 150: 0


## 3. Save Clean Chunks

In [4]:
clean_output = {
    'created_at': datetime.now().isoformat(),
    'total_chunks': len(clean_chunks),
    'cleaning': {
        'toc_removed': toc_removed,
        'cover_removed': cover_removed,
        'trimmed': trimmed,
    },
    'chunks': clean_chunks,
}

clean_path = CHUNKS_DIR / 'clean_chunks.json'
clean_path.write_text(json.dumps(clean_output, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'clean_chunks.json saved -> {clean_path}')
print(f'Size: {clean_path.stat().st_size / (1024*1024):.1f} MB')

clean_chunks.json saved -> D:\Cybernetic Gym Assistant\backend\data\processed\chunks\clean_chunks.json
Size: 4.9 MB


## 4. Load BGE-M3 Embedding Model

In [7]:
import torch

print('Loading BAAI/bge-m3...')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

embed_model = BGEM3FlagModel(
    EMBEDDING_MODEL,
    use_fp16=torch.cuda.is_available()
)
print('Model loaded OK')
print(f'Embedding dim: {EMBED_DIM}')

Loading BAAI/bge-m3...
Device: cpu


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model loaded OK
Embedding dim: 1024


## 5. Generate Embeddings

In [8]:
texts = [c['text'] for c in clean_chunks]

print(f'Embedding {len(texts):,} chunks in batches of {BATCH_SIZE}...')
print('Estimated time: ~{:.0f} min on CPU'.format(len(texts) / BATCH_SIZE / 60 * 8))

all_embeddings = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc='Embedding'):
    batch = texts[i : i + BATCH_SIZE]
    out = embed_model.encode(
        batch,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )
    all_embeddings.append(out['dense_vecs'])

embeddings = np.vstack(all_embeddings).astype('float32')

print(f'\nEmbeddings shape: {embeddings.shape}')
print(f'Expected: ({len(clean_chunks)}, {EMBED_DIM})')
assert embeddings.shape == (len(clean_chunks), EMBED_DIM), 'Shape mismatch!'
print('Shape OK')

Embedding 2,098 chunks in batches of 16...
Estimated time: ~17 min on CPU


Embedding: 100%|██████████| 132/132 [2:13:43<00:00, 60.79s/it]


Embeddings shape: (2098, 1024)
Expected: (2098, 1024)
Shape OK


## 6. Normalize + Build FAISS Index

In [9]:
# L2-normalize for cosine similarity via inner product
faiss.normalize_L2(embeddings)

# IndexFlatIP = exact inner product search (cosine on normalized vectors)
# For production with >100K chunks, switch to IndexIVFFlat or HNSW
index = faiss.IndexFlatIP(EMBED_DIM)
index.add(embeddings)

print(f'FAISS index built')
print(f'  Index type: IndexFlatIP (exact cosine)')
print(f'  Vectors: {index.ntotal:,}')
print(f'  Dimensions: {EMBED_DIM}')

# Save index
index_path = VS_DIR / 'henselmans.index'
faiss.write_index(index, str(index_path))
print(f'  Saved -> {index_path}')
print(f'  Size   : {index_path.stat().st_size / (1024*1024):.1f} MB')

FAISS index built
  Index type: IndexFlatIP (exact cosine)
  Vectors: 2,098
  Dimensions: 1024
  Saved -> D:\Cybernetic Gym Assistant\backend\data\vectorstore\henselmans.index
  Size   : 8.2 MB


## 7. Save Metadata (aligned with FAISS index)

In [10]:
# Metadata array must be in EXACT same order as vectors in FAISS
# FAISS index returns integer IDs → we look up metadata[id]
metadata = []
for i, c in enumerate(clean_chunks):
    metadata.append({
        'faiss_id': i,
        'chunk_id': c['chunk_id'],
        'source': c['source'],
        'file_type': c['file_type'],
        'text': c['text'],
        'topic_category': c['metadata']['topic_category'],
        'applies_to_level': c['metadata']['applies_to_level'],
        'goal_relevance': c['metadata']['goal_relevance'],
        'is_case_study': c['metadata']['is_case_study'],
        'calculator_context': c['metadata']['calculator_context'],
        'is_calculator_tool': c['metadata'].get('is_calculator_tool', False),
    })

meta_path = VS_DIR / 'henselmans_metadata.json'
meta_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'Metadata saved -> {meta_path}')
print(f'Size: {meta_path.stat().st_size / (1024*1024):.1f} MB')
print(f'Records: {len(metadata):,}')
print(f'\nSanity check: metadata[0] chunk_id = {metadata[0]["chunk_id"]}')
print(f'Sanity check: metadata[-1] chunk_id = {metadata[-1]["chunk_id"]}')

Metadata saved -> D:\Cybernetic Gym Assistant\backend\data\vectorstore\henselmans_metadata.json
Size: 4.7 MB
Records: 2,098

Sanity check: metadata[0] chunk_id = AAS PTC 2022__0004
Sanity check: metadata[-1] chunk_id = calculator__training_volume__description


## 8. Retrieval Smoke Test

In [11]:
def retrieve(query: str, k: int = 5, filter_meta: dict = None) -> list[dict]:
    """
    Basic retrieval function.
    filter_meta: optional dict of metadata key->value to filter results.
    Example: filter_meta={'topic_category': 'nutrition', 'applies_to_level': 'novice'}
    """
    # Encode query
    q_emb = embed_model.encode(
        [query],
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )['dense_vecs'].astype('float32')
    faiss.normalize_L2(q_emb)

    # Search - retrieve more if filtering
    search_k = k * 5 if filter_meta else k
    scores, ids = index.search(q_emb, search_k)

    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue
        meta = metadata[idx]

        # Apply metadata filter
        if filter_meta:
            if not all(meta.get(key) == val for key, val in filter_meta.items()):
                continue

        results.append({**meta, 'score': float(score)})
        if len(results) >= k:
            break

    return results


# Test 1: English query (standard)
print('TEST 1: English query - protein intake for muscle building')
print('-' * 55)
results = retrieve('How much protein do I need to build muscle?', k=3)
for r in results:
    print(f'  [{r["score"]:.4f}] [{r["topic_category"]}] {r["source"][:45]}')
    print(f'           {r["text"][:120]}...')
    print()

# Test 2: Bulgarian query (cross-lingual - no translation yet)
print('TEST 2: Bulgarian query (cross-lingual test)')
print('-' * 55)
results = retrieve('Колко протеин трябва да приемам за мускулна маса?', k=3)
for r in results:
    print(f'  [{r["score"]:.4f}] [{r["topic_category"]}] {r["source"][:45]}')
    print(f'           {r["text"][:120]}...')
    print()

# Test 3: Filtered retrieval (novice + training)
print('TEST 3: Filtered - novice training volume')
print('-' * 55)
results = retrieve('training volume for beginners', k=3,
                   filter_meta={'topic_category': 'training', 'applies_to_level': 'novice'})
for r in results:
    print(f'  [{r["score"]:.4f}] level={r["applies_to_level"]} | {r["source"][:40]}')
    print(f'           {r["text"][:120]}...')
    print()

# Test 4: Case study retrieval
print('TEST 4: Case study retrieval')
print('-' * 55)
results = retrieve('intermediate male cutting program design', k=3,
                   filter_meta={'is_case_study': True})
for r in results:
    print(f'  [{r["score"]:.4f}] {r["source"][:50]}')
    print(f'           {r["text"][:120]}...')
    print()

TEST 1: English query - protein intake for muscle building
-------------------------------------------------------
  [0.7416] [nutrition] Protein PTC 2022.pdf
           els.

A big-picture overview of protein metabolism. The protein from our diet can be
metabolized to obtain amino acids o...

  [0.7364] [nutrition] Protein PTC 2022.pdf
           cle growth: you also
need an actual stimulus for it, such as strength training.

The muscle-full effect (source)


--- P...

  [0.7278] [nutrition] Protein PTC 2022.pdf
           Pea-rice-mix (7:3)
7 %
9 g
Take-home message
To maximize protein balance, it is advisable to consume at least 0.3 g/kg p...

TEST 2: Bulgarian query (cross-lingual test)
-------------------------------------------------------
  [0.7246] [nutrition] Protein PTC 2022.pdf
           o et al. (2020) found no difference in body composition changes between
similar Mediterranean diets with an evenish 17-3...

  [0.7232] [nutrition] Protein PTC 2022.pdf
           Pea-rice-

## 9. Final Summary

In [12]:
print('=' * 60)
print('  NOTEBOOK 05 — COMPLETE')
print('=' * 60)
print(f'  Clean chunks: {len(clean_chunks):,}')
print(f'  FAISS vectors: {index.ntotal:,}')
print(f'  Embedding model: {EMBEDDING_MODEL} ({EMBED_DIM}d)')
print(f'  Index type: IndexFlatIP (exact cosine)')
print()
print(f'  Files saved:')
print(f'    data/processed/chunks/clean_chunks.json')
print(f'    data/vectorstore/henselmans.index')
print(f'    data/vectorstore/henselmans_metadata.json')

  NOTEBOOK 05 — COMPLETE
  Clean chunks: 2,098
  FAISS vectors: 2,098
  Embedding model: BAAI/bge-m3 (1024d)
  Index type: IndexFlatIP (exact cosine)

  Files saved:
    data/processed/chunks/clean_chunks.json
    data/vectorstore/henselmans.index
    data/vectorstore/henselmans_metadata.json
